In [1]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func
import torch.optim as optim
from torch.amp import GradScaler, autocast

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)

        self.initial = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool
        )
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4

    def forward(self, x):
        x0 = self.initial(x)
        x1 = self.encoder1(x0)
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x1)

        out = self.final_conv(d1)
        return out

In [2]:
from torch.utils.data import Dataset
import tifffile as tif
import os
from torchvision import transforms
import random

random.seed(42)

imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
maskTransform = transforms.Compose([transforms.ToTensor()])

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (maskTransform(mask) > 0).float()

        return img, mask
    
def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

path = "../data/"
regions = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]

subdataset_to_region = {
    "Hokkaido Iburi-Tobu": "Hokkaido Iburi-Tobu",
    "Jiuzhai valley (UAV-0.2m)": "Jiuzhai Valley",
    "Jiuzhai valley (UAV-0.5m)": "Jiuzhai Valley",
    "Lombok": "Lombok",
    "Longxi River (SAT)": "Longxi River",
    "Longxi River (UAV)": "Longxi River",
    "Mengdong Township": "Mengdong Township",
    "Moxi town (UAV-0.2m)": "Luding",
    "Moxi town (UAV-1m)": "Luding",
    "Moxitaidi (SAT)": "Luding",
    "Moxitaidi (UAV-0.6m)": "Luding",
    "Moxitaidi (UAV-1m)": "Luding",
    "palu": "Palu",
    "Tiburon Peninsula (planet)": "Tiburon Peninsula",
    "Tiburon Peninsula (Sentinel)": "Tiburon Peninsula",
    "Wenchuan": "Wenchuan"
}

regions_dict = {
    "Hokkaido Iburi-Tobu": [],
    "Jiuzhai Valley": [],
    "Lombok": [],
    "Longxi River": [],
    "Mengdong Township": [],
    "Luding": [],
    "Palu": [],
    "Tiburon Peninsula": [],
    "Wenchuan": []
}

for region in regions:
    if(region != "study areas shp"):
        dataset_dir = "../data/" + region
        image_dir = os.path.join(dataset_dir, "img")
        img_list = os.listdir(image_dir)
        
        all_images = sorted(os.path.join(image_dir, f) for f in img_list)
        
        regions_dict[subdataset_to_region[region]].extend(all_images)
        
for region in regions_dict:
    if(len(regions_dict[region]) > 1000):
        regions_dict[region] = random.sample(regions_dict[region], 1000)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))

pos_weight = torch.tensor([4.5]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
### TARGET-ONLY (UPPER BOUND)

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

batch_size = 16

output = "region,precision,recall,f1,iou,miou,oa"

for region in regions_dict:    
    image_paths = regions_dict[region]
    mask_paths = [f.replace("img", "mask") for f in image_paths]
    
    train_img, temp_img, train_mask, temp_mask = train_test_split(
        image_paths, mask_paths, test_size=.3, random_state=42
    )
    
    test_img, val_img, test_mask, val_mask = train_test_split(
        temp_img, temp_mask, test_size=.5, random_state=42
    )

    train_dataset = LandslideDataset(train_img, train_mask)
    val_dataset = LandslideDataset(val_img, val_mask)
    test_dataset = LandslideDataset(test_img, test_mask)
    
    trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)
    
    epochs = 40
    
    best_iou = 0.0
    patience = 10
    counter = 0
        
    model_path = "../models/supervised/intra-region/" + region + ".pth"
    
    model = ResNetUNet()
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    scaler = GradScaler()
    
    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa')
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_num = 0

        for i, data in enumerate(trainLoader, 0):
            image, mask = data
            
            image = image.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()
            
            with autocast(device_type="cuda"):
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                bce = criterion(outputs, mask)
                dice = dice_loss(outputs, mask)
                loss = bce + dice
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            train_num += 1
        
        model.eval()
        val_loss = 0.0
        val_num = 0
        
        TP, FP, FN, TN = 0,0,0,0
        
        with torch.no_grad():
            for data in valLoader:
                image, mask = data

                image = image.to(device)
                mask = mask.to(device)
            
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                loss = criterion(outputs, mask)
                val_loss += loss.item()

                preds = torch.sigmoid(outputs) > .6
                
                preds_flat = preds.view(-1)
                mask_flat = mask.view(-1)
                
                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
        
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
        
        print(f'{region}, {epoch+1}, {running_loss / train_num :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    
    TP, FP, FN, TN = 0,0,0,0
    
    for data in testLoader:
        image, mask = data

        image = image.to(device)
        mask = mask.to(device)
            
        outputs = model(image)
        outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)

        preds = torch.sigmoid(outputs) > .6
                
        preds_flat = preds.view(-1)
        mask_flat = mask.view(-1)
                
        TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
        FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
        FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
        TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
        
    precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
    output += f'\n{region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'

with open("../results/supervised/intra-region.csv", "w") as f:
    f.write(output)

region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, 1, 1.645, 0.769, 0.7347, 0.0118, 0.0231, 0.0117, 0.4562, 0.9008
Hokkaido Iburi-Tobu, 2, 0.913, 0.357, 0.5905, 0.8238, 0.6879, 0.5243, 0.7214, 0.9253
Hokkaido Iburi-Tobu, 3, 0.728, 0.311, 0.6783, 0.8110, 0.7387, 0.5857, 0.7616, 0.9426
Hokkaido Iburi-Tobu, 4, 0.669, 0.313, 0.7093, 0.7907, 0.7478, 0.5972, 0.7696, 0.9467
Hokkaido Iburi-Tobu, 5, 0.706, 0.312, 0.6919, 0.8009, 0.7424, 0.5904, 0.7650, 0.9444
Hokkaido Iburi-Tobu, 6, 0.662, 0.298, 0.7144, 0.8081, 0.7584, 0.6108, 0.7774, 0.9485
Hokkaido Iburi-Tobu, 7, 0.678, 0.280, 0.6087, 0.8895, 0.7227, 0.5659, 0.7455, 0.9318
Hokkaido Iburi-Tobu, 8, 0.634, 0.268, 0.6701, 0.8560, 0.7517, 0.6022, 0.7702, 0.9435
Hokkaido Iburi-Tobu, 9, 0.618, 0.278, 0.7237, 0.8205, 0.7691, 0.6248, 0.7856, 0.9507
Hokkaido Iburi-Tobu, 10, 0.622, 0.264, 0.6270, 0.8917, 0.7363, 0.5827, 0.7563, 0.9361
Hokkaido Iburi-Tobu, 11, 0.601, 0.267, 0.6964, 0.8473, 0.7645, 0.6188, 0.78

In [ ]:
### SOURCE ONLY (LOWER BOUND)

from torch.utils.data import DataLoader

batch_size = 16

model = ResNetUNet()
model.to(device)
model.eval()

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

for source_region in regions_dict:
    model_path = "../models/supervised/intra-region/" + source_region + ".pth"
    model.load_state_dict(torch.load(model_path, map_location=device))
    
    print(f'source_region, target_region, precision, recall, f1, iou, miou, oa')
    
    for target_region in regions_dict:
        if source_region != target_region:
        
            TP, FP, FN, TN = 0,0,0,0
                        
            image_paths = regions_dict[target_region]
            mask_paths = [f.replace("img", "mask") for f in image_paths]
            
            test_dataset = LandslideDataset(image_paths, mask_paths)
            testLoader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
            for data in testLoader:
                image, mask = data

                image = image.to(device)
                mask = mask.to(device)
                    
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)

                preds = torch.sigmoid(outputs) > .6
                        
                preds_flat = preds.view(-1)
                mask_flat = mask.view(-1)
                        
                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
        
            precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
            output += f'\n{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'
            print(f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')


with open("../results/supervised/0_tuning.csv", "w") as f:
    f.write(output)

source_region, target_region, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, Jiuzhai Valley, 0.8403, 0.5236, 0.6452, 0.4762, 0.6645, 0.8702
Hokkaido Iburi-Tobu, Lombok, 0.0972, 0.4426, 0.1594, 0.0866, 0.4698, 0.8549
Hokkaido Iburi-Tobu, Longxi River, 0.4491, 0.0524, 0.0939, 0.0493, 0.4681, 0.8876
Hokkaido Iburi-Tobu, Mengdong Township, 0.5457, 0.1858, 0.2772, 0.1609, 0.5321, 0.9049
Hokkaido Iburi-Tobu, Luding, 0.3682, 0.4049, 0.3857, 0.2389, 0.5463, 0.8602
Hokkaido Iburi-Tobu, Palu, 0.1087, 0.3239, 0.1628, 0.0886, 0.5241, 0.9597
Hokkaido Iburi-Tobu, Tiburon Peninsula, 0.1980, 0.1969, 0.1975, 0.1095, 0.5174, 0.9260
Hokkaido Iburi-Tobu, Wenchuan, 0.6182, 0.2409, 0.3467, 0.2097, 0.5824, 0.9556
source_region, target_region, precision, recall, f1, iou, miou, oa
Jiuzhai Valley, Hokkaido Iburi-Tobu, 0.5627, 0.4061, 0.4718, 0.3087, 0.6097, 0.9142
Jiuzhai Valley, Lombok, 0.0784, 0.4439, 0.1333, 0.0714, 0.4447, 0.8205
Jiuzhai Valley, Longxi River, 0.5715, 0.1925, 0.2880, 0.1682, 0.530

In [ ]:
# FINE-TUNED 5%

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

batch_size = 8

percent = 100

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

for source_region in regions_dict:
    model_path = "../models/supervised/intra-region/" + source_region + ".pth"
    
    for target_region in regions_dict:
        if source_region != target_region:
            model = ResNetUNet()
            model.to(device)            
            model.load_state_dict(torch.load(model_path, map_location=device))
        
            image_paths = regions_dict[target_region]
            mask_paths = [f.replace("img", "mask") for f in image_paths]
            
            train_img, test_img, train_mask, test_mask = train_test_split(
                image_paths, mask_paths, test_size=.2, random_state=42
            )
            
            ft_images = random.sample(train_img, int(percent / 100 * len(train_img)))
            ft_masks = [f.replace("img", "mask") for f in ft_images]
            
            ft_train_img, ft_val_img, ft_train_mask, ft_val_mask = train_test_split(
                ft_images, ft_masks, test_size=.2, random_state=42
            )
                    
            train_dataset = LandslideDataset(ft_train_img, ft_train_mask)
            val_dataset = LandslideDataset(ft_val_img, ft_val_mask)
            test_dataset = LandslideDataset(test_img, test_mask)
            
            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)
    
            epochs = 40
    
            best_iou = 0.0
            patience = 10
            counter = 0
        
            save_path = f"../models/supervised/fine-tuning/{percent}/{percent}_{source_region}_{target_region}.pth"
            
            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            scaler = GradScaler()
            
            print(f'source_region, target_region, precision, recall, f1, iou, miou, oa')
            
            for epoch in range(epochs):
                model.train()
                running_loss = 0.0
                train_num = 0

                for i, data in enumerate(trainLoader, 0):
                    image, mask = data
                    
                    image = image.to(device)
                    mask = mask.to(device)

                    optimizer.zero_grad()
                    
                    with autocast(device_type="cuda"):
                        outputs = model(image)
                        outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                        bce = criterion(outputs, mask)
                        dice = dice_loss(outputs, mask)
                        loss = bce + dice
                    
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()

                    running_loss += loss.item()
                    train_num += 1
                
                model.eval()
                val_loss = 0.0
                val_num = 0
                
                TP, FP, FN, TN = 0,0,0,0
                
                with torch.no_grad():
                    for data in valLoader:
                        image, mask = data

                        image = image.to(device)
                        mask = mask.to(device)
                    
                        outputs = model(image)
                        outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                        loss = criterion(outputs, mask)
                        val_loss += loss.item()

                        preds = torch.sigmoid(outputs) > .6
                        
                        preds_flat = preds.view(-1)
                        mask_flat = mask.view(-1)
                        
                        TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                        FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                        FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                        TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                        
                        val_num += 1
                
                precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
                
                print(f'{source_region}, {target_region}, {epoch+1}, {running_loss / train_num :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')

                if iou > best_iou:
                    best_iou = iou
                    counter = 0
                    
                    torch.save(model.state_dict(), save_path)
                elif iou != 0.0:
                    counter += 1
                    if counter >= patience:
                        break
            
            model.load_state_dict(torch.load(save_path, map_location=device))
            
            TP, FP, FN, TN = 0,0,0,0
            
            for data in testLoader:
                image, mask = data

                image = image.to(device)
                mask = mask.to(device)
                    
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)

                preds = torch.sigmoid(outputs) > .6

                preds_flat = preds.view(-1)
                mask_flat = mask.view(-1)
                        
                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
            precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
            output += f'\n{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'

with open(f"../results/supervised/{percent}_tuning.csv", "w") as f:
    f.write(output)

source_region, target_region, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, Jiuzhai Valley, 1, 2.265, 0.609, 0.6142, 0.8612, 0.7170, 0.5589, 0.6886, 0.8523
Hokkaido Iburi-Tobu, Jiuzhai Valley, 2, 0.933, 0.519, 0.7160, 0.8447, 0.7750, 0.6327, 0.7511, 0.8934
Hokkaido Iburi-Tobu, Jiuzhai Valley, 3, 0.879, 0.552, 0.6852, 0.8569, 0.7615, 0.6148, 0.7358, 0.8834
Hokkaido Iburi-Tobu, Jiuzhai Valley, 4, 0.835, 0.514, 0.7707, 0.8249, 0.7969, 0.6623, 0.7755, 0.9086
Hokkaido Iburi-Tobu, Jiuzhai Valley, 5, 0.795, 0.690, 0.7643, 0.7898, 0.7769, 0.6351, 0.7581, 0.9014
Hokkaido Iburi-Tobu, Jiuzhai Valley, 6, 0.797, 0.502, 0.7171, 0.8696, 0.7860, 0.6475, 0.7603, 0.8971
Hokkaido Iburi-Tobu, Jiuzhai Valley, 7, 0.690, 0.533, 0.7709, 0.8376, 0.8029, 0.6707, 0.7807, 0.9106
Hokkaido Iburi-Tobu, Jiuzhai Valley, 8, 0.644, 0.578, 0.6944, 0.8792, 0.7760, 0.6339, 0.7488, 0.8897
Hokkaido Iburi-Tobu, Jiuzhai Valley, 9, 0.565, 0.786, 0.7667, 0.7765, 0.7716, 0.6281, 0.7540, 0.9001
Hokkaido Iburi-Tobu, Jiu